In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Hello World: Twój pierwszy obwód kwantowy

Zbuduj [stan Bella](https://en.wikipedia.org/wiki/Bell_state) (dwa splątane qubity) i uruchom go na trzy sposoby:

1. **Symulacja idealna** — doskonałe wyniki, konto nie jest wymagane
2. **Symulacja z szumem** — symuluje prawdziwe urządzenie, konto nie jest wymagane
3. **Prawdziwy sprzęt kwantowy** — wymaga bezpłatnego konta IBM Quantum (kroki konfiguracji poniżej)

## Zbuduj obwód

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

qc.draw(output="mpl")

## Opcja 1: Symulacja idealna (konto nie jest wymagane)
Używa `StatevectorSampler` — lokalnego symulatora dającego doskonałe, wolne od szumu wyniki.

In [ ]:
from qiskit.primitives import StatevectorSampler

result = StatevectorSampler().run([qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(counts)

## Opcja 2: Symulacja z szumem (konto nie jest wymagane)
Używa `FakeManilaV2` — lokalnego symulatora odwzorowującego prawdziwe urządzenie kwantowe IBM, włącznie z jego charakterystyką szumu. Obwód musi najpierw zostać poddany transpilacji (adaptacji) do zestawu bramek i topologii qubitów danego urządzenia.

In [ ]:
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = FakeManilaV2()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Opcja 3: Prawdziwy sprzęt kwantowy
Wymaga bezpłatnego konta IBM Quantum. Aby je założyć:

1. Zarejestruj się pod adresem [quantum.cloud.ibm.com/registration](https://quantum.cloud.ibm.com/registration) — przez pierwsze 30 dni karta kredytowa nie jest wymagana
2. Zaloguj się pod adresem [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) i wybierz region **us-east** (wymagany dla bezpłatnego planu Open Plan)
3. Utwórz instancję (bezpłatny Open Plan) w sekcji [Instances](https://quantum.cloud.ibm.com/instances), jeśli jeszcze jej nie masz
4. Utwórz klucz API pod adresem [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) (lub pod adresem [cloud.ibm.com/iam/apikeys](https://cloud.ibm.com/iam/apikeys))
5. Skopiuj swój **CRN** (Cloud Resource Name) ze strony [Instances](https://quantum.cloud.ibm.com/instances)

Jeśli nie zapisałeś jeszcze swoich danych uwierzytelniających w tej sesji Binder, uruchom poniższą komórkę. Zastąp `<your-api-key>` kluczem API z kroku 4, a `<your-crn>` — wartością CRN z kroku 5.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="<your-api-key>",
    instance="<your-crn>",
    set_as_default=True,
    overwrite=True,
)

**Uwaga:** Zadania na prawdziwym sprzęcie mogą zająć trochę czasu w zależności od kolejki. Jeśli komórka wciąż działa, możesz sprawdzić status zadania i zobaczyć wyniki na stronie [quantum.cloud.ibm.com/workloads](https://quantum.cloud.ibm.com/workloads?user=me).

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on {backend.name}")

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

result = SamplerV2(mode=backend).run([isa_qc], shots=1024).result()
counts = result[0].data.meas.get_counts()
counts

In [ ]:
plot_histogram(counts)

## Co dalej?
- **[Samouczki](https://doqumentation.org/tutorials)** — przewodniki krok po kroku dotyczące algorytmów, mitygacji błędów, transpilacji i nie tylko
- **[Poradniki](https://doqumentation.org/guides)** — materiały referencyjne dotyczące uruchamiania obwodów, prymitywów i środowiska Qiskit Runtime
- **[Kursy](https://doqumentation.org/learning/courses/basics-of-quantum-information)** — ustrukturyzowane ścieżki nauki od podstaw kwantowych po obliczenia w skali użytkowej
- **[Moduły](https://doqumentation.org/learning/modules/computer-science)** — pogłębione moduły koncepcyjne z zakresu informatyki i mechaniki kwantowej